In [1]:
# Notebook 02: Data Cleaning
# Author: Eric Lindolfo, B.Eng
# Goal: Clean and prepare Grocery Inventory dataset for analysis

import pandas as pd
import numpy as np

# Load raw dataset
inventory = pd.read_csv('data/Grocery_Inventory_new_v1.csv')

print("Dataset loaded successfully!")
print("Shape:", inventory.shape)

Dataset loaded successfully!
Shape: (990, 17)


In [2]:
# Step 1: Standardize column names
# Fix typo: 'Catagory' → 'Category'
# Best practice: lowercase, no spaces, consistent naming

inventory.columns = inventory.columns.str.strip()

inventory = inventory.rename(columns={'Catagory': 'Category'})

print("Column names after cleaning:")
print(inventory.columns.tolist())

Column names after cleaning:
['Product_Name', 'Category', 'Supplier_Name', 'Warehouse_Location', 'Status', 'Product_ID', 'Supplier_ID', 'Date_Received', 'Last_Order_Date', 'Expiration_Date', 'Stock_Quantity', 'Reorder_Level', 'Reorder_Quantity', 'Unit_Price', 'Sales_Volume', 'Inventory_Turnover_Rate', 'percentage']


In [3]:
# Step 2: Convert date columns from object (text) to datetime
# This allows pandas to calculate days between dates
# pd.to_datetime() → tells pandas "this column is a date"

date_columns = ['Date_Received', 'Last_Order_Date', 'Expiration_Date']

for col in date_columns:
    inventory[col] = pd.to_datetime(inventory[col])

print("Date columns converted successfully!")
print("\nData types after conversion:")
print(inventory[date_columns].dtypes)

Date columns converted successfully!

Data types after conversion:
Date_Received      datetime64[ns]
Last_Order_Date    datetime64[ns]
Expiration_Date    datetime64[ns]
dtype: object


In [4]:
# Step 3: Clean Unit_Price — remove '$' and convert to float
# Step 4: Clean percentage — remove '%' and convert to float

inventory['Unit_Price'] = inventory['Unit_Price'].str.replace('$', '', regex=False).astype(float)

inventory['percentage'] = inventory['percentage'].str.replace('%', '', regex=False).astype(float)

print("Unit_Price and percentage cleaned successfully!")
print("\nData types:")
print(inventory[['Unit_Price', 'percentage']].dtypes)
print("\nSample values:")
print(inventory[['Unit_Price', 'percentage']].head())

Unit_Price and percentage cleaned successfully!

Data types:
Unit_Price    float64
percentage    float64
dtype: object

Sample values:
   Unit_Price  percentage
0         4.6        1.96
1         2.0        0.91
2        12.0        1.36
3         1.5        1.36
4         7.0        2.17


In [5]:
# Step 5: Investigate and handle the 1 null value in Category
# First let's find which row has the null value

null_category = inventory[inventory['Category'].isnull()]
print("Row with null Category:")
print(null_category[['Product_Name', 'Category', 'Supplier_Name', 'Status']])

Row with null Category:
    Product_Name Category Supplier_Name        Status
572      Cabbage      NaN         Rooxo  Discontinued


In [6]:
# Step 6: Fill null Category for Cabbage
# We can confidently infer the category from the product name

inventory.loc[inventory['Product_Name'] == 'Cabbage', 'Category'] = 'Fruits & Vegetables'

# Verify the fix
print("Null values in Category after fix:")
print(inventory['Category'].isnull().sum())

print("\nVerifying Cabbage row:")
print(inventory[inventory['Product_Name'] == 'Cabbage'][['Product_Name', 'Category']])

Null values in Category after fix:
0

Verifying Cabbage row:
    Product_Name             Category
70       Cabbage  Fruits & Vegetables
220      Cabbage  Fruits & Vegetables
224      Cabbage  Fruits & Vegetables
458      Cabbage  Fruits & Vegetables
572      Cabbage  Fruits & Vegetables
749      Cabbage  Fruits & Vegetables
961      Cabbage  Fruits & Vegetables
987      Cabbage  Fruits & Vegetables


In [7]:
# Step 7: Feature Engineering
# Create analytical columns needed for Q1, Q2, and Q4

import datetime

# Today's date — reference point for all calculations
today = pd.Timestamp.today().normalize()

# days_to_expiry → how many days until the product expires
# Negative = already expired
inventory['days_to_expiry'] = (inventory['Expiration_Date'] - today).dt.days

# days_of_supply → how many days current stock will last
# Stock_Quantity divided by average daily sales (Sales_Volume / 30)
inventory['days_of_supply'] = (inventory['Stock_Quantity'] / (inventory['Sales_Volume'] / 30)).round(1)

print("Feature engineering completed!")
print("\nSample — days_to_expiry and days_of_supply:")
print(inventory[['Product_Name', 'Expiration_Date', 'Stock_Quantity', 'Sales_Volume', 'days_to_expiry', 'days_of_supply']].head(10))

Feature engineering completed!

Sample — days_to_expiry and days_of_supply:
      Product_Name Expiration_Date  Stock_Quantity  Sales_Volume  \
0      Bell Pepper      2025-01-31              46            96   
1    Vegetable Oil      2024-06-11              51            24   
2  Parmesan Cheese      2024-04-08              38            35   
3           Carrot      2024-09-26              51            44   
4           Garlic      2024-05-20              27            91   
5            Lemon      2024-08-06              91            38   
6    Coconut Sugar      2024-03-30              17            76   
7        Anchovies      2024-08-22              81            95   
8           Cheese      2024-03-07              78            60   
9           Yogurt      2024-10-25              55            62   

   days_to_expiry  days_of_supply  
0            -457            14.4  
1            -691            63.8  
2            -755            32.6  
3            -584            34

In [8]:
# Step 8: Create expiry_risk_flag
# Classifies each product by risk level based on
# days_to_expiry vs days_of_supply

def classify_risk(row):
    if row['days_to_expiry'] < 0:
        return 'EXPIRED'
    elif row['days_to_expiry'] <= row['days_of_supply']:
        return 'HIGH'
    elif row['days_to_expiry'] <= row['days_of_supply'] * 2:
        return 'MEDIUM'
    else:
        return 'LOW'

inventory['expiry_risk_flag'] = inventory.apply(classify_risk, axis=1)

print("Risk classification complete!")
print("\nRisk distribution:")
print(inventory['expiry_risk_flag'].value_counts())

Risk classification complete!

Risk distribution:
expiry_risk_flag
EXPIRED    990
Name: count, dtype: int64


In [9]:
# Step 9: Adjust expiration dates for demonstration purposes
# Dataset uses 2024 dates — adding 2 years to simulate current inventory
# This adjustment is documented in project limitations

from dateutil.relativedelta import relativedelta

inventory['Expiration_Date_Adjusted'] = inventory['Expiration_Date'] + pd.DateOffset(years=2)

# Recalculate days_to_expiry with adjusted dates
inventory['days_to_expiry'] = (inventory['Expiration_Date_Adjusted'] - today).dt.days

# Recalculate expiry_risk_flag
inventory['expiry_risk_flag'] = inventory.apply(classify_risk, axis=1)

print("Dates adjusted successfully!")
print("\nRisk distribution after adjustment:")
print(inventory['expiry_risk_flag'].value_counts())

print("\nSample:")
print(inventory[['Product_Name', 'Expiration_Date_Adjusted', 'days_to_expiry', 'days_of_supply', 'expiry_risk_flag']].head(10))

Dates adjusted successfully!

Risk distribution after adjustment:
expiry_risk_flag
LOW        616
EXPIRED    188
HIGH       116
MEDIUM      70
Name: count, dtype: int64

Sample:
      Product_Name Expiration_Date_Adjusted  days_to_expiry  days_of_supply  \
0      Bell Pepper               2027-01-31             273            14.4   
1    Vegetable Oil               2026-06-11              39            63.8   
2  Parmesan Cheese               2026-04-08             -25            32.6   
3           Carrot               2026-09-26             146            34.8   
4           Garlic               2026-05-20              17             8.9   
5            Lemon               2026-08-06              95            71.8   
6    Coconut Sugar               2026-03-30             -34             6.7   
7        Anchovies               2026-08-22             111            25.6   
8           Cheese               2026-03-07             -57            39.0   
9           Yogurt              

In [10]:
# Step 10: Export cleaned dataset for analysis and Power BI
# Save to data/processed/ folder

import os

# Create processed folder if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

inventory.to_csv('data/processed/inventory_clean.csv', index=False)

print("Clean dataset exported successfully!")
print("Location: data/processed/inventory_clean.csv")
print("\nFinal shape:", inventory.shape)
print("Columns:", inventory.columns.tolist())

Clean dataset exported successfully!
Location: data/processed/inventory_clean.csv

Final shape: (990, 21)
Columns: ['Product_Name', 'Category', 'Supplier_Name', 'Warehouse_Location', 'Status', 'Product_ID', 'Supplier_ID', 'Date_Received', 'Last_Order_Date', 'Expiration_Date', 'Stock_Quantity', 'Reorder_Level', 'Reorder_Quantity', 'Unit_Price', 'Sales_Volume', 'Inventory_Turnover_Rate', 'percentage', 'days_to_expiry', 'days_of_supply', 'expiry_risk_flag', 'Expiration_Date_Adjusted']
